# Финальное решение

Запускаешь все ячейки по порядку - получаешь output/submission.csv.

Что внутри:
- собираем ~165 фич (см. features.py)
- три модели: LightGBM, CatBoost, XGBoost
- ансамбль через rank-mean, веса подбираются на time-val

Средний скор на time-val: P@R0.7 около 0.95.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd

sys.path.insert(0, '..')
from features import build_xy, events_in_window
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
VALID_FROM = '2026-04-17'

train = pd.read_csv('../data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('../data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('../data/events.csv.gz', parse_dates=['event_ts'])
ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print('train:', len(train), '| test:', len(test))

train: 11091 | test: 4909


In [ ]:
# фичи
Xtr, Xte, ytr = build_xy(ev_tr, ev_te, train, test)
print('Xtr:', Xtr.shape, '| Xte:', Xte.shape, '| pos rate:', round(ytr.mean(), 4))

is_val = (train.set_index('cookie_id').loc[Xtr.cookie_id, 'window_start_ts'] >= VALID_FROM).values
print('train_fit:', (~is_val).sum(), '| valid:', is_val.sum())

Xtr: (11091, 166) | Xte: (4909, 166) | pos rate: 0.0811
train_fit: 9140 | valid: 1951


In [ ]:
# метрика
from metric import precision_at_recall

def evaluate(y, score):
    return {
        'p_at_r07': precision_at_recall(y, score, 0.7),
        'roc_auc': roc_auc_score(y, score),
        'pr_auc': average_precision_score(y, score),
    }

y_val = ytr.values[is_val]

In [ ]:
# обучаем три модели
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

cols = [c for c in Xtr.columns if c != 'cookie_id']
Xt = Xtr[cols]
Xv = Xtr.loc[is_val, cols]
Xfit = Xtr.loc[~is_val, cols]
Xte_ = Xte[cols]
y_fit = ytr.values[~is_val]
y_all = ytr.values

lgbm = LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                      min_child_samples=8, subsample=0.9, colsample_bytree=0.8,
                      reg_lambda=1.0, random_state=SEED, verbose=-1)
lgbm.fit(Xfit, y_fit); p_va_l = lgbm.predict_proba(Xv)[:, 1]
lgbm.fit(Xt, y_all); p_te_l = lgbm.predict_proba(Xte_)[:, 1]

cb = CatBoostClassifier(iterations=800, learning_rate=0.05, depth=6,
                        l2_leaf_reg=3.0, random_seed=SEED,
                        verbose=False, allow_writing_files=False)
cb.fit(Xfit, y_fit); p_va_c = cb.predict_proba(Xv)[:, 1]
cb.fit(Xt, y_all); p_te_c = cb.predict_proba(Xte_)[:, 1]

xgb = XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=6,
                    min_child_weight=3, subsample=0.9, colsample_bytree=0.8,
                    reg_lambda=1.0, random_state=SEED, verbosity=0,
                    eval_metric='logloss', tree_method='hist')
xgb.fit(Xfit, y_fit); p_va_x = xgb.predict_proba(Xv)[:, 1]
xgb.fit(Xt, y_all); p_te_x = xgb.predict_proba(Xte_)[:, 1]

print('LGBM    :', evaluate(y_val, p_va_l))
print('CatBoost:', evaluate(y_val, p_va_c))
print('XGBoost :', evaluate(y_val, p_va_x))

LGBM    : {'p_at_r07': 0.9495798319327731, 'roc_auc': 0.991890005583473, 'pr_auc': 0.9173176609288167}
CatBoost: {'p_at_r07': 0.928, 'roc_auc': 0.9923401730876605, 'pr_auc': 0.9232148658004841}
XGBoost : {'p_at_r07': 0.9523809523809523, 'roc_auc': 0.9920121440536014, 'pr_auc': 0.9190713490137914}


In [ ]:
# ансамбль через rank-mean, веса подбираем на валидации
def rank01(s):
    return pd.Series(s).rank(method='average').values / (len(s) + 1)

best = (0, None)
for wl in [0, 1]:
    for wc in [0, 1, 2]:
        for wx in [0, 1, 2]:
            if wl + wc + wx == 0:
                continue
            v = (wl * rank01(p_va_l) + wc * rank01(p_va_c) + wx * rank01(p_va_x)) / (wl + wc + wx)
            p = precision_at_recall(y_val, v)
            if p > best[0]:
                best = (p, (wl, wc, wx))


wl, wc, wx = best[1]
print('лучшие веса L=%d C=%d X=%d -> P@R0.7 = %.4f' % (wl, wc, wx, best[0]))

test_blend = (wl * rank01(p_te_l) + wc * rank01(p_te_c) + wx * rank01(p_te_x)) / (wl + wc + wx)
val_blend = (wl * rank01(p_va_l) + wc * rank01(p_va_c) + wx * rank01(p_va_x)) / (wl + wc + wx)
print('итого P@R0.7:', round(precision_at_recall(y_val, val_blend), 4))

лучшие веса L=0 C=0 X=1 -> P@R0.7 = 0.9524
итого P@R0.7: 0.9524


In [ ]:
# сохраняем submission
import os
os.makedirs('../output', exist_ok=True)

sub = pd.DataFrame({'cookie_id': Xte.cookie_id.values, 'score': test_blend})
sub.to_csv('../output/submission.csv', index=False)

# sanity checks
sample = pd.read_csv('../sample_submission.csv')
assert sub.shape == sample.shape
assert list(sub.columns) == list(sample.columns)
assert set(sub.cookie_id) == set(sample.cookie_id)
assert sub.cookie_id.is_unique
assert sub.score.between(0, 1).all()
print('OK:', len(sub), 'строк, score в [%.4f, %.4f]' % (sub.score.min(), sub.score.max()))
sub.head()

OK: 4909 строк, score в [0.0002, 0.9998]


,cookie_id,score
0,ck_315fb710a0e371e7,0.272505
1,ck_a76ee3b3e3e522fd,0.647454
2,ck_94c9a4d382689e82,0.942159
3,ck_8eaf9509ad9462a0,0.038289
4,ck_9a88a5a989cb5bc6,0.900204
